In [24]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from llmsource import llm
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

model = "qwen3.5:9b"
 

# # llm = ChatOllama(
# #     model=model,
# #     temperature=0
# # )
# class ChatState(TypedDict):
#     messages: Annotated[list[BaseMessage],add_messages]


# def chat_node(state:ChatState)->ChatState:
#     message = state['messages']
   
#     response = llm_with_tools.invoke(message)
    
#     return{
#         "messages":[response]
#     }
# checkpoint = MemorySaver()
# graph = StateGraph(ChatState)
# graph.add_node('chat_node',chat_node)

# # define edges
# graph.add_edge(START,'chat_node')
# graph.add_edge('chat_node',END)
# chatbot = graph.compile(checkpoint)

# # memory
# config = {
#     "configurable": {
#         "thread_id": "1"
#     }
# }



# init={"messages": [HumanMessage(content="My name is user 2")]}

# response = workflow.invoke(init,config)
# print(response['messages'][-1].content)        

In [25]:
import numexpr

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        result = numexpr.evaluate(expression)
        return str(result)
    except Exception as e:
        return f"Error: {e}"


In [ ]:
from typing import Literal , TypedDict ,Annotated
from langgraph.graph import StateGraph , START, END
from langchain_ollama import ChatOllama
from pydantic import Field, BaseModel
from langgraph.graph.message import BaseMessage , add_messages
from langchain_core.messages import HumanMessage ,AIMessage
from langgraph.checkpoint.memory import MemorySaver
from llmsource import llm
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool

model = "qwen3.5:9b"
tools = [calculator]
tools_node = ToolNode(tools)
llm_with_tools = llm.bind_tools(tools)

# llm = ChatOllama(
#     model=model,
#     temperature=0
# )
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]


def chat_node(state:ChatState)->ChatState:
    message = state['messages']
   
    response = llm_with_tools.invoke(message)
    
    return{
        "messages":[response]
    }
checkpoint = MemorySaver()
graph = StateGraph(ChatState)
graph.add_node('chat_node',chat_node)
graph.add_node('tools',tools_node)

graph.add_conditional_edges('chat_node',tools_condition)

# define edges
graph.add_edge(START,'chat_node')
graph.add_edge('tools_node','chat_node')

# graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpoint)

# memory
config = {
    "configurable": {
        "thread_id": "1"
    }
}




    

In [42]:
init={"messages": [HumanMessage(content="what is current weather in cairo egypt")]}

response = chatbot.invoke(init,config)
print(response['messages'][-1].content)    

I don't have access to real-time data, so I cannot tell you the current weather in Cairo, Egypt right now. You can check a weather website or app for the most up-to-date information.
